# RQ5 — 0DTE SPX Options Backtest

**Research question:**  
*Can these predictions be used to construct a profitable end-of-day options trading strategy after accounting for transaction costs and realistic execution constraints?*

This notebook is the economic backtest stage of the dissertation.

It consumes the option contracts and minute bars produced by:

`rq5_options_data_retrieval_extended_otm.ipynb`

and evaluates whether the already-developed RQ1–RQ3 signals translate into option-level profitability.

## Primary strategy — fixed before option P&L is examined

Candidate session:

- RQ1 confidence in the top 30%;
- RQ2 predicted absolute move ≥ 20 bps;
- RQ3 large-movement signal = 1.

Direction:

- RQ1 predicts Up → buy a Call;
- RQ1 predicts Down → buy a Put.

Primary contract:

- nearest available **ATM 0DTE SPX option**.

Sensitivity contracts:

- approximately 5, 10, 15, 20, 25 and 30 SPX points OTM.

## Required benchmark

The 60-minute mean-reversion rule is evaluated on the **same candidate dates**, with the same strike-selection and execution assumptions. This is important because it outperformed the selected RQ1 ML model on unconditional directional balanced accuracy.

## Execution design

The underlying feature set uses completed information through the 14:59 ET minute. To avoid same-bar look-ahead:

- signal decision point: 15:00 ET;
- entry reference: **open of the first available option minute bar from 15:00–15:05 ET**;
- exit reference: **close of the last available option minute bar from 15:55–15:59 ET**.

If the required entry or exit window contains no trade-derived aggregate bar, that contract/date is excluded rather than having a price invented.

## Historical quote limitation

The Options Starter data used here contain trade-derived minute aggregates rather than historical NBBO bid/ask quotes. Therefore the exact historical spread cannot be reconstructed.

The notebook reports several **synthetic adverse-execution scenarios**. These should be interpreted as cost sensitivity tests, not reconstructed historical fills.

No fresh post-development holdout is used.

## 1. Imports and backtest configuration

In [ ]:
from pathlib import Path
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42

# ------------------------------------------------------------
# Frozen trading rule
# ------------------------------------------------------------
PRIMARY_RQ1_COVERAGE = 0.30
PRIMARY_RQ2_MIN_BPS = 20.0

# RQ2 thresholds above 20 are subsets of the primary candidate set and are
# therefore used only as sensitivity analyses.
RQ2_THRESHOLD_SENSITIVITY_BPS = [20, 25, 30]

# Primary and retail-affordability strike-distance tests.
STRIKE_OFFSETS_POINTS = [0, 5, 10, 15, 20, 25, 30]
PRIMARY_STRIKE_OFFSET = 0

# ------------------------------------------------------------
# Deterministic entry/exit rules
# ------------------------------------------------------------
ENTRY_WINDOW_START = "15:00"
ENTRY_WINDOW_END = "15:05"
EXIT_WINDOW_START = "15:55"
EXIT_WINDOW_END = "15:59"

ENTRY_REFERENCE_FIELD = "open"
EXIT_REFERENCE_FIELD = "close"

# If contract metadata are incomplete, SPX-style contracts conventionally
# use a $100 multiplier. Every fallback is explicitly flagged in the trade log.
DEFAULT_CONTRACT_MULTIPLIER = 100.0

# ------------------------------------------------------------
# Synthetic execution sensitivity
# ------------------------------------------------------------
# For each side of a trade:
# penalty_points = max(min_penalty_points, pct_of_reference_price * reference_price)
#
# Buy:  executed entry = reference entry + penalty
# Sell: executed exit  = max(0, reference exit - penalty)
#
# commission_per_side_usd is a generic commission/fee sensitivity assumption,
# not a claim about a specific broker.
EXECUTION_SCENARIOS = {
    "frictionless": {
        "min_penalty_points": 0.00,
        "pct_of_reference_price": 0.000,
        "commission_per_side_usd": 0.00,
    },
    "low_cost": {
        "min_penalty_points": 0.05,
        "pct_of_reference_price": 0.020,
        "commission_per_side_usd": 1.00,
    },
    "medium_cost": {
        "min_penalty_points": 0.10,
        "pct_of_reference_price": 0.050,
        "commission_per_side_usd": 1.50,
    },
    "severe_cost": {
        "min_penalty_points": 0.20,
        "pct_of_reference_price": 0.100,
        "commission_per_side_usd": 2.00,
    },
}

PRIMARY_COST_SCENARIO = "medium_cost"

# Descriptive capital thresholds only. Edit these if a specific retail-account
# budget is to be studied.
AFFORDABILITY_THRESHOLDS_USD = [250, 500, 1000, 2000]

# "Near-total loss" is defined as losing at least 80% of premium at risk.
NEAR_TOTAL_LOSS_RETURN = -0.80

MIN_REGIME_TRADES = 5

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 2. Locate and load RQ5 retrieval outputs

In [ ]:
def locate_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "market.duckdb").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/market.duckdb. "
        "Run this notebook inside your dissertation project."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())

RQ5_ROOT = PROJECT_ROOT / "outputs" / "rq5_options_trading"
RQ5_RAW_ROOT = RQ5_ROOT / "raw"
RQ5_TABLE_ROOT = RQ5_ROOT / "tables"
RQ5_FIGURE_ROOT = RQ5_ROOT / "figures"
RQ5_BACKTEST_ROOT = RQ5_ROOT / "backtest"

RQ5_TABLE_ROOT.mkdir(parents=True, exist_ok=True)
RQ5_FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
RQ5_BACKTEST_ROOT.mkdir(parents=True, exist_ok=True)

BARS_PATH = RQ5_RAW_ROOT / "rq5_option_minute_bars.parquet"
CONTRACT_SELECTION_PATH = RQ5_TABLE_ROOT / "rq5_contract_selection.csv"
CANDIDATES_PATH = RQ5_TABLE_ROOT / "rq5_primary_candidate_sessions.csv"
QUALITY_PATH = RQ5_TABLE_ROOT / "rq5_option_data_quality.csv"
RETRIEVAL_MANIFEST_PATH = RQ5_ROOT / "rq5_retrieval_manifest.json"

required = [
    BARS_PATH,
    CONTRACT_SELECTION_PATH,
    CANDIDATES_PATH,
    QUALITY_PATH,
]
missing = [str(path) for path in required if not path.exists()]

if missing:
    raise FileNotFoundError(
        "Run rq5_options_data_retrieval_extended_otm.ipynb first.\n"
        "Missing:\n- " + "\n- ".join(missing)
    )

bars = pd.read_parquet(BARS_PATH)
selection = pd.read_csv(CONTRACT_SELECTION_PATH)
candidates = pd.read_csv(CANDIDATES_PATH)
quality = pd.read_csv(QUALITY_PATH)

for frame in [bars, selection, candidates, quality]:
    if "session_date" in frame.columns:
        frame["session_date"] = pd.to_datetime(frame["session_date"])

bars["timestamp"] = pd.to_datetime(bars["timestamp"])

# The retrieval file may contain duplicate bars because the same listed contract
# can satisfy multiple target strike offsets. Keep one raw bar per contract/time.
bars_unique = (
    bars.sort_values(["session_date", "option_ticker", "timestamp"])
    .drop_duplicates(
        subset=["session_date", "option_ticker", "timestamp"],
        keep="first",
    )
    .reset_index(drop=True)
)

selection = selection[
    selection["selection_status"].eq("selected")
].copy()

selection["strike_price"] = pd.to_numeric(
    selection["strike_price"],
    errors="coerce",
)
selection["otm_offset_points"] = pd.to_numeric(
    selection["otm_offset_points"],
    errors="coerce",
)

print("Unique option minute bars:", len(bars_unique))
print("Selected contract rows:", len(selection))
print("Primary candidate sessions:", len(candidates))
print("Date range:", candidates["session_date"].min(), "to", candidates["session_date"].max())

if RETRIEVAL_MANIFEST_PATH.exists():
    retrieval_manifest = json.loads(
        RETRIEVAL_MANIFEST_PATH.read_text(encoding="utf-8")
    )
    display(pd.Series(retrieval_manifest, name="retrieval_manifest").to_frame())

## 3. Reconfirm the candidate-date rule

The retrieval notebook already froze the primary candidate dates. This cell checks that the required signal columns are present and reconstructs the ML and mean-reversion directions without looking at option returns.

In [ ]:
required_candidate_cols = {
    "session_date",
    "spx_at_1500",
    "rq1_prediction",
    "rq1_confidence_percentile",
    "rq2_predicted_move_bps",
    "rq3_positive",
    "ret_last_60m",
}

missing_candidate_cols = required_candidate_cols.difference(candidates.columns)
if missing_candidate_cols:
    raise ValueError(
        f"Candidate file missing columns: {sorted(missing_candidate_cols)}"
    )

candidates["rq1_high_confidence"] = (
    candidates["rq1_confidence_percentile"] > (1 - PRIMARY_RQ1_COVERAGE)
)

candidates["reconstructed_primary_candidate"] = (
    candidates["rq1_high_confidence"]
    & (candidates["rq2_predicted_move_bps"] >= PRIMARY_RQ2_MIN_BPS)
    & candidates["rq3_positive"].astype(bool)
)

if not candidates["reconstructed_primary_candidate"].all():
    raise ValueError(
        "Candidate file contains rows that no longer satisfy the frozen "
        "30%-confidence / RQ2>=20bps / RQ3-positive rule."
    )

candidates["ml_direction"] = np.where(
    candidates["rq1_prediction"].astype(int).eq(1),
    "call",
    "put",
)

candidates["mean_reversion_direction"] = np.where(
    candidates["ret_last_60m"] < 0,
    "call",
    "put",
)

print("Candidate rule verified.")
display(
    candidates[
        [
            "session_date",
            "rq1_confidence_percentile",
            "rq2_predicted_move_bps",
            "rq3_positive",
            "ml_direction",
            "mean_reversion_direction",
        ]
    ]
)

## 4. Extract deterministic entry and exit reference prices

For each selected contract:

- entry = open of first available bar between 15:00 and 15:05;
- exit = close of last available bar between 15:55 and 15:59.

No interpolation is used.

A contract is backtest-eligible only when both references exist and are strictly positive.

In [ ]:
def hhmm(series: pd.Series) -> pd.Series:
    return series.dt.strftime("%H:%M")


def choose_reference_bars(contract_bars: pd.DataFrame):
    contract_bars = contract_bars.sort_values("timestamp").copy()
    time_text = hhmm(contract_bars["timestamp"])

    entry = contract_bars[
        (time_text >= ENTRY_WINDOW_START)
        & (time_text <= ENTRY_WINDOW_END)
    ].head(1)

    exit_ = contract_bars[
        (time_text >= EXIT_WINDOW_START)
        & (time_text <= EXIT_WINDOW_END)
    ].tail(1)

    if entry.empty or exit_.empty:
        return None

    entry_row = entry.iloc[0]
    exit_row = exit_.iloc[0]

    entry_price = pd.to_numeric(
        pd.Series([entry_row[ENTRY_REFERENCE_FIELD]]),
        errors="coerce",
    ).iloc[0]

    exit_price = pd.to_numeric(
        pd.Series([exit_row[EXIT_REFERENCE_FIELD]]),
        errors="coerce",
    ).iloc[0]

    if (
        not np.isfinite(entry_price)
        or not np.isfinite(exit_price)
        or entry_price <= 0
        or exit_price < 0
    ):
        return None

    return {
        "entry_timestamp": entry_row["timestamp"],
        "exit_timestamp": exit_row["timestamp"],
        "entry_reference_price": float(entry_price),
        "exit_reference_price": float(exit_price),
        "entry_bar_volume": float(entry_row.get("volume", np.nan)),
        "exit_bar_volume": float(exit_row.get("volume", np.nan)),
        "entry_bar_vwap": float(entry_row.get("vwap", np.nan))
        if pd.notna(entry_row.get("vwap", np.nan))
        else np.nan,
        "exit_bar_vwap": float(exit_row.get("vwap", np.nan))
        if pd.notna(exit_row.get("vwap", np.nan))
        else np.nan,
    }


reference_rows = []

for row in selection.itertuples(index=False):
    contract_bars = bars_unique[
        bars_unique["session_date"].eq(pd.Timestamp(row.session_date))
        & bars_unique["option_ticker"].eq(row.ticker)
    ].copy()

    ref = choose_reference_bars(contract_bars)

    base = {
        "session_date": pd.Timestamp(row.session_date),
        "option_ticker": row.ticker,
        "option_type": str(row.option_type).lower(),
        "otm_offset_points": float(row.otm_offset_points),
        "strike_price": float(row.strike_price),
        "spot_for_selection": float(row.spot_for_selection),
        "shares_per_contract": getattr(row, "shares_per_contract", np.nan),
        "reference_available": ref is not None,
    }

    if ref is not None:
        base.update(ref)

    reference_rows.append(base)

contract_refs = pd.DataFrame(reference_rows)

print("Contract selections:", len(contract_refs))
print("Usable entry/exit references:", int(contract_refs["reference_available"].sum()))
display(
    contract_refs.groupby(
        ["otm_offset_points", "option_type", "reference_available"]
    ).size().rename("contracts").to_frame()
)

## 5. Create strategy-level trades before costs

In [ ]:
signal_cols = [
    "session_date",
    "spx_at_1500",
    "rq1_confidence_percentile",
    "rq2_predicted_move_bps",
    "rq3_positive",
    "ml_direction",
    "mean_reversion_direction",
]

candidate_signals = candidates[signal_cols].copy()

strategy_frames = []

for strategy_name, direction_col in [
    ("RQ1_ML", "ml_direction"),
    ("MeanReversion60m", "mean_reversion_direction"),
]:
    frame = candidate_signals.copy()
    frame["strategy"] = strategy_name
    frame["required_option_type"] = frame[direction_col]

    joined = frame.merge(
        contract_refs[
            contract_refs["reference_available"]
        ],
        left_on=["session_date", "required_option_type"],
        right_on=["session_date", "option_type"],
        how="left",
        validate="one_to_many",
    )

    strategy_frames.append(joined)

gross_trades = pd.concat(strategy_frames, ignore_index=True)

gross_trades = gross_trades[
    gross_trades["otm_offset_points"].isin(STRIKE_OFFSETS_POINTS)
].copy()

gross_trades["contract_multiplier"] = pd.to_numeric(
    gross_trades["shares_per_contract"],
    errors="coerce",
)

gross_trades["multiplier_fallback_used"] = (
    gross_trades["contract_multiplier"].isna()
    | (gross_trades["contract_multiplier"] <= 0)
)

gross_trades.loc[
    gross_trades["multiplier_fallback_used"],
    "contract_multiplier",
] = DEFAULT_CONTRACT_MULTIPLIER

gross_trades["gross_option_change_points"] = (
    gross_trades["exit_reference_price"]
    - gross_trades["entry_reference_price"]
)

gross_trades["gross_pnl_usd"] = (
    gross_trades["gross_option_change_points"]
    * gross_trades["contract_multiplier"]
)

gross_trades["gross_return_on_premium"] = (
    gross_trades["gross_pnl_usd"]
    / (
        gross_trades["entry_reference_price"]
        * gross_trades["contract_multiplier"]
    )
)

gross_trades = gross_trades.sort_values(
    ["strategy", "otm_offset_points", "session_date"]
).reset_index(drop=True)

print("Gross strategy-contract observations:", len(gross_trades))
display(
    gross_trades[
        [
            "session_date",
            "strategy",
            "required_option_type",
            "otm_offset_points",
            "strike_price",
            "entry_timestamp",
            "entry_reference_price",
            "exit_timestamp",
            "exit_reference_price",
            "gross_pnl_usd",
            "gross_return_on_premium",
        ]
    ].head(40)
)

## 6. Apply synthetic execution-cost scenarios

Because historical quotes are unavailable, these are deliberately explicit sensitivity assumptions.

The adverse execution penalty is applied to **both** entry and exit:

- entry price is increased;
- exit price is reduced but never below zero;
- commission/fee sensitivity is charged once on entry and once on exit.

The frictionless case is only a reference upper bound, not the main conclusion.

In [ ]:
def apply_execution_scenario(frame: pd.DataFrame, scenario_name: str, params: dict):
    out = frame.copy()

    min_penalty = float(params["min_penalty_points"])
    pct_penalty = float(params["pct_of_reference_price"])
    commission_side = float(params["commission_per_side_usd"])

    out["entry_execution_penalty_points"] = np.maximum(
        min_penalty,
        pct_penalty * out["entry_reference_price"],
    )

    out["exit_execution_penalty_points"] = np.maximum(
        min_penalty,
        pct_penalty * out["exit_reference_price"],
    )

    out["entry_execution_price"] = (
        out["entry_reference_price"]
        + out["entry_execution_penalty_points"]
    )

    out["exit_execution_price"] = np.maximum(
        0.0,
        out["exit_reference_price"]
        - out["exit_execution_penalty_points"],
    )

    out["commission_entry_usd"] = commission_side
    out["commission_exit_usd"] = commission_side
    out["total_commission_usd"] = 2 * commission_side

    out["capital_at_risk_usd"] = (
        out["entry_execution_price"]
        * out["contract_multiplier"]
        + out["commission_entry_usd"]
    )

    out["net_pnl_usd"] = (
        (
            out["exit_execution_price"]
            - out["entry_execution_price"]
        )
        * out["contract_multiplier"]
        - out["total_commission_usd"]
    )

    out["net_return_on_premium"] = (
        out["net_pnl_usd"]
        / out["capital_at_risk_usd"]
    )

    out["execution_scenario"] = scenario_name

    out["winner"] = out["net_pnl_usd"] > 0
    out["near_total_loss"] = (
        out["net_return_on_premium"] <= NEAR_TOTAL_LOSS_RETURN
    )

    return out


net_frames = []

for scenario_name, params in EXECUTION_SCENARIOS.items():
    net_frames.append(
        apply_execution_scenario(
            gross_trades,
            scenario_name,
            params,
        )
    )

trades = pd.concat(net_frames, ignore_index=True)

display(
    trades[
        [
            "session_date",
            "strategy",
            "otm_offset_points",
            "execution_scenario",
            "entry_reference_price",
            "entry_execution_price",
            "exit_reference_price",
            "exit_execution_price",
            "capital_at_risk_usd",
            "net_pnl_usd",
            "net_return_on_premium",
        ]
    ].head(40)
)

## 7. Performance-summary helpers

In [ ]:
def profit_factor(pnl: pd.Series) -> float:
    gains = pnl[pnl > 0].sum()
    losses = -pnl[pnl < 0].sum()

    if losses == 0:
        return np.inf if gains > 0 else np.nan

    return float(gains / losses)


def max_drawdown_from_pnl(frame: pd.DataFrame) -> float:
    ordered = frame.sort_values("session_date")
    equity = ordered["net_pnl_usd"].cumsum()
    peak = equity.cummax()
    drawdown = equity - peak
    return float(drawdown.min()) if len(drawdown) else np.nan


def sharpe_style(returns: pd.Series) -> float:
    returns = pd.to_numeric(returns, errors="coerce").dropna()
    if len(returns) < 2 or returns.std(ddof=1) == 0:
        return np.nan
    # Non-annualised per-trade Sharpe-style statistic.
    return float(returns.mean() / returns.std(ddof=1))


def sortino_style(returns: pd.Series) -> float:
    returns = pd.to_numeric(returns, errors="coerce").dropna()
    downside = returns[returns < 0]

    if len(returns) < 2 or len(downside) == 0:
        return np.nan

    downside_dev = np.sqrt(np.mean(np.square(downside)))
    if downside_dev == 0:
        return np.nan

    # Non-annualised per-trade Sortino-style statistic.
    return float(returns.mean() / downside_dev)


def summarize_trades(frame: pd.DataFrame) -> dict:
    if frame.empty:
        return {
            "trades": 0,
            "win_rate": np.nan,
            "total_pnl_usd": np.nan,
            "mean_pnl_usd": np.nan,
            "median_pnl_usd": np.nan,
            "mean_return_on_premium": np.nan,
            "median_return_on_premium": np.nan,
            "profit_factor": np.nan,
            "max_drawdown_usd": np.nan,
            "sharpe_style_per_trade": np.nan,
            "sortino_style_per_trade": np.nan,
            "median_capital_at_risk_usd": np.nan,
            "mean_capital_at_risk_usd": np.nan,
            "near_total_loss_rate": np.nan,
        }

    return {
        "trades": int(len(frame)),
        "win_rate": float((frame["net_pnl_usd"] > 0).mean()),
        "total_pnl_usd": float(frame["net_pnl_usd"].sum()),
        "mean_pnl_usd": float(frame["net_pnl_usd"].mean()),
        "median_pnl_usd": float(frame["net_pnl_usd"].median()),
        "mean_return_on_premium": float(frame["net_return_on_premium"].mean()),
        "median_return_on_premium": float(frame["net_return_on_premium"].median()),
        "profit_factor": profit_factor(frame["net_pnl_usd"]),
        "max_drawdown_usd": max_drawdown_from_pnl(frame),
        "sharpe_style_per_trade": sharpe_style(frame["net_return_on_premium"]),
        "sortino_style_per_trade": sortino_style(frame["net_return_on_premium"]),
        "median_capital_at_risk_usd": float(frame["capital_at_risk_usd"].median()),
        "mean_capital_at_risk_usd": float(frame["capital_at_risk_usd"].mean()),
        "near_total_loss_rate": float(frame["near_total_loss"].mean()),
    }

## 8. Core strategy results: ML vs mean reversion across strike distances and costs

In [ ]:
summary_rows = []

for keys, subset in trades.groupby(
    ["strategy", "otm_offset_points", "execution_scenario"],
    dropna=False,
):
    strategy, offset, scenario = keys
    metrics = summarize_trades(subset)

    summary_rows.append(
        {
            "strategy": strategy,
            "otm_offset_points": offset,
            "execution_scenario": scenario,
            **metrics,
        }
    )

strategy_summary = pd.DataFrame(summary_rows).sort_values(
    ["execution_scenario", "otm_offset_points", "strategy"]
)

display(strategy_summary)

summary_path = RQ5_TABLE_ROOT / "rq5_strategy_summary.csv"
strategy_summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)

## 9. Primary dissertation comparison

The primary RQ5 specification is:

- candidate rule = 30% RQ1 confidence + RQ2 ≥ 20 bps + RQ3 positive;
- contract = ATM;
- execution sensitivity = medium-cost scenario.

The frictionless case is shown as a reference only.

This table compares the ML direction against the 60-minute mean-reversion direction on the same opportunity dates.

In [ ]:
primary_comparison = strategy_summary[
    strategy_summary["otm_offset_points"].eq(PRIMARY_STRIKE_OFFSET)
    & strategy_summary["execution_scenario"].isin(
        ["frictionless", PRIMARY_COST_SCENARIO]
    )
].copy()

display(primary_comparison)

primary_path = RQ5_TABLE_ROOT / "rq5_primary_atm_comparison.csv"
primary_comparison.to_csv(primary_path, index=False)
print("Saved:", primary_path)

## 10. Retail affordability and capital-efficiency analysis

In [ ]:
affordability_rows = []

medium = trades[
    trades["execution_scenario"].eq(PRIMARY_COST_SCENARIO)
].copy()

for keys, subset in medium.groupby(
    ["strategy", "otm_offset_points"],
    dropna=False,
):
    strategy, offset = keys

    row = {
        "strategy": strategy,
        "otm_offset_points": offset,
        "trades": len(subset),
        "median_entry_capital_usd": subset["capital_at_risk_usd"].median(),
        "mean_entry_capital_usd": subset["capital_at_risk_usd"].mean(),
        "p25_entry_capital_usd": subset["capital_at_risk_usd"].quantile(0.25),
        "p75_entry_capital_usd": subset["capital_at_risk_usd"].quantile(0.75),
        "mean_net_pnl_usd": subset["net_pnl_usd"].mean(),
        "median_net_pnl_usd": subset["net_pnl_usd"].median(),
        "mean_return_on_premium": subset["net_return_on_premium"].mean(),
        "win_rate": (subset["net_pnl_usd"] > 0).mean(),
        "near_total_loss_rate": subset["near_total_loss"].mean(),
        "profit_factor": profit_factor(subset["net_pnl_usd"]),
    }

    for threshold in AFFORDABILITY_THRESHOLDS_USD:
        row[f"share_entry_capital_le_{threshold}"] = (
            subset["capital_at_risk_usd"] <= threshold
        ).mean()

    affordability_rows.append(row)

affordability = pd.DataFrame(affordability_rows).sort_values(
    ["strategy", "otm_offset_points"]
)

display(affordability)

affordability_path = RQ5_TABLE_ROOT / "rq5_retail_affordability.csv"
affordability.to_csv(affordability_path, index=False)
print("Saved:", affordability_path)

## 11. Strike-distance sensitivity

This directly addresses whether cheaper farther-OTM contracts improve accessibility without creating unacceptable losses in win rate or P&L.

Do **not** select the best-looking strike after the fact as the new primary strategy. ATM remains the primary specification; farther-OTM strikes are sensitivity evidence.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for strategy, subset in affordability.groupby("strategy"):
    axes[0, 0].plot(
        subset["otm_offset_points"],
        subset["median_entry_capital_usd"],
        marker="o",
        label=strategy,
    )
    axes[0, 1].plot(
        subset["otm_offset_points"],
        subset["mean_return_on_premium"],
        marker="o",
        label=strategy,
    )
    axes[1, 0].plot(
        subset["otm_offset_points"],
        subset["win_rate"],
        marker="o",
        label=strategy,
    )
    axes[1, 1].plot(
        subset["otm_offset_points"],
        subset["near_total_loss_rate"],
        marker="o",
        label=strategy,
    )

axes[0, 0].set_title("Median capital required")
axes[0, 0].set_xlabel("SPX points OTM")
axes[0, 0].set_ylabel("USD")

axes[0, 1].set_title("Mean return on premium")
axes[0, 1].set_xlabel("SPX points OTM")
axes[0, 1].set_ylabel("Return")

axes[1, 0].set_title("Win rate")
axes[1, 0].set_xlabel("SPX points OTM")
axes[1, 0].set_ylabel("Win rate")

axes[1, 1].set_title("Near-total loss rate")
axes[1, 1].set_xlabel("SPX points OTM")
axes[1, 1].set_ylabel("Share of trades")

for ax in axes.ravel():
    ax.legend()
    ax.grid(alpha=0.2)

fig.suptitle(
    f"RQ5 Retail-Affordability Sensitivity — {PRIMARY_COST_SCENARIO}",
    y=1.02,
)

fig.tight_layout()
figure_path = RQ5_FIGURE_ROOT / "rq5_otm_affordability_sensitivity.png"
fig.savefig(figure_path, dpi=220, bbox_inches="tight")
print("Saved:", figure_path)
plt.show()

## 12. Transaction-cost sensitivity

In [ ]:
cost_pivot = strategy_summary.pivot_table(
    index=["strategy", "otm_offset_points"],
    columns="execution_scenario",
    values=[
        "total_pnl_usd",
        "mean_return_on_premium",
        "win_rate",
        "profit_factor",
    ],
)

display(cost_pivot)

cost_path = RQ5_TABLE_ROOT / "rq5_transaction_cost_sensitivity.csv"
cost_pivot.reset_index().to_csv(cost_path, index=False)
print("Saved:", cost_path)

## 13. RQ2 opportunity-threshold sensitivity

The 25- and 30-bps thresholds were **not** used to redefine the primary strategy after seeing RQ4. They are reported here only as sensitivity subsets of the already-retrieved ≥20-bps candidate sample.

In [ ]:
threshold_rows = []

for threshold in RQ2_THRESHOLD_SENSITIVITY_BPS:
    subset_all = medium[
        medium["rq2_predicted_move_bps"] >= threshold
    ].copy()

    for keys, subset in subset_all.groupby(
        ["strategy", "otm_offset_points"],
        dropna=False,
    ):
        strategy, offset = keys

        threshold_rows.append(
            {
                "rq2_min_predicted_move_bps": threshold,
                "strategy": strategy,
                "otm_offset_points": offset,
                **summarize_trades(subset),
            }
        )

threshold_sensitivity = pd.DataFrame(threshold_rows)

display(threshold_sensitivity)

threshold_path = RQ5_TABLE_ROOT / "rq5_rq2_threshold_sensitivity.csv"
threshold_sensitivity.to_csv(threshold_path, index=False)
print("Saved:", threshold_path)

## 14. Regime analysis

In [ ]:
regime_columns = [
    col
    for col in [
        "vix_regime",
        "realized_vol_regime",
        "intraday_trend_regime",
    ]
    if col in medium.columns
]

regime_rows = []

primary_medium = medium[
    medium["otm_offset_points"].eq(PRIMARY_STRIKE_OFFSET)
].copy()

for regime_col in regime_columns:
    for keys, subset in primary_medium.groupby(
        ["strategy", regime_col],
        dropna=False,
    ):
        strategy, regime = keys

        regime_rows.append(
            {
                "regime_type": regime_col,
                "regime": regime,
                "strategy": strategy,
                "trades": len(subset),
                "minimum_sample_met": len(subset) >= MIN_REGIME_TRADES,
                **summarize_trades(subset),
            }
        )

regime_results = pd.DataFrame(regime_rows)

if len(regime_results):
    display(regime_results)
    regime_path = RQ5_TABLE_ROOT / "rq5_regime_results.csv"
    regime_results.to_csv(regime_path, index=False)
    print("Saved:", regime_path)
else:
    print(
        "Regime columns were not present in the retrieved candidate metadata. "
        "This section is skipped."
    )

## 15. Chronological P&L and drawdown

In [ ]:
equity_frames = []

plot_subset = medium[
    medium["otm_offset_points"].isin([0, 10, 20, 30])
].copy()

for keys, subset in plot_subset.groupby(
    ["strategy", "otm_offset_points"],
    dropna=False,
):
    strategy, offset = keys

    ordered = subset.sort_values("session_date").copy()
    ordered["cumulative_pnl_usd"] = ordered["net_pnl_usd"].cumsum()
    ordered["running_peak_usd"] = ordered["cumulative_pnl_usd"].cummax()
    ordered["drawdown_usd"] = (
        ordered["cumulative_pnl_usd"]
        - ordered["running_peak_usd"]
    )
    ordered["series_name"] = f"{strategy} | {int(offset)}pt OTM"
    equity_frames.append(ordered)

equity_curves = pd.concat(equity_frames, ignore_index=True)

fig, ax = plt.subplots(figsize=(12, 6))

for name, subset in equity_curves.groupby("series_name"):
    ax.plot(
        subset["session_date"],
        subset["cumulative_pnl_usd"],
        marker="o",
        linewidth=1.5,
        label=name,
    )

ax.axhline(0, color="black", linewidth=1, linestyle="--")
ax.set_title(
    f"RQ5 Cumulative P&L — {PRIMARY_COST_SCENARIO}"
)
ax.set_xlabel("Session date")
ax.set_ylabel("Cumulative P&L (USD)")
ax.legend(ncol=2, fontsize=8)
ax.grid(alpha=0.2)

fig.tight_layout()
equity_path = RQ5_FIGURE_ROOT / "rq5_cumulative_pnl_medium_cost.png"
fig.savefig(equity_path, dpi=220, bbox_inches="tight")
print("Saved:", equity_path)
plt.show()

## 16. Trade-level paired ML vs mean-reversion comparison

Because the ML and mean-reversion strategies trade the same opportunity dates and strike offsets, their dollar P&L can be compared date-by-date.

This is more informative than comparing two unrelated trade samples.

In [ ]:
paired_source = medium.copy()

paired = paired_source.pivot_table(
    index=["session_date", "otm_offset_points"],
    columns="strategy",
    values="net_pnl_usd",
    aggfunc="first",
).reset_index()

required_strategy_cols = {"RQ1_ML", "MeanReversion60m"}

if required_strategy_cols.issubset(paired.columns):
    paired = paired.dropna(
        subset=["RQ1_ML", "MeanReversion60m"]
    ).copy()

    paired["ml_minus_mean_reversion_pnl_usd"] = (
        paired["RQ1_ML"]
        - paired["MeanReversion60m"]
    )

    paired_summary = (
        paired.groupby("otm_offset_points")
        .agg(
            paired_dates=("session_date", "size"),
            mean_ml_minus_benchmark_usd=(
                "ml_minus_mean_reversion_pnl_usd",
                "mean",
            ),
            median_ml_minus_benchmark_usd=(
                "ml_minus_mean_reversion_pnl_usd",
                "median",
            ),
            share_ml_outperforms=(
                "ml_minus_mean_reversion_pnl_usd",
                lambda x: float((x > 0).mean()),
            ),
        )
        .reset_index()
    )

    display(paired_summary)

    paired_path = RQ5_TABLE_ROOT / "rq5_paired_ml_vs_mean_reversion.csv"
    paired.to_csv(paired_path, index=False)

    paired_summary_path = (
        RQ5_TABLE_ROOT / "rq5_paired_ml_vs_mean_reversion_summary.csv"
    )
    paired_summary.to_csv(paired_summary_path, index=False)

    print("Saved:", paired_path)
    print("Saved:", paired_summary_path)
else:
    print("Insufficient overlapping ML and mean-reversion trade records.")

## 17. Strategy-level bootstrap uncertainty

The number of option trades may be small. This section therefore uses a moving-block bootstrap over chronological trade dates for the **primary ATM medium-cost specification**.

These intervals describe uncertainty within the available development-period backtest. They are not a substitute for the future untouched holdout.

In [ ]:
N_BOOTSTRAPS = 2_000
BOOTSTRAP_BLOCK_LENGTH = 3
CONFIDENCE_LEVEL = 0.95


def moving_block_indices(n, block_length, rng):
    if n <= 0:
        return np.array([], dtype=int)

    block_length = max(1, min(block_length, n))
    starts = np.arange(0, n - block_length + 1)

    result = []
    while len(result) < n:
        start = int(rng.choice(starts))
        result.extend(range(start, start + block_length))

    return np.asarray(result[:n], dtype=int)


def ci(values, confidence_level=0.95):
    values = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()

    if len(values) == 0:
        return np.nan, np.nan

    alpha = 1 - confidence_level
    return (
        float(values.quantile(alpha / 2)),
        float(values.quantile(1 - alpha / 2)),
    )


bootstrap_rows = []
rng = np.random.default_rng(RANDOM_STATE)

primary_boot = primary_medium.sort_values(
    ["strategy", "session_date"]
)

for strategy, subset in primary_boot.groupby("strategy"):
    subset = subset.sort_values("session_date").reset_index(drop=True)

    draws = []

    for iteration in range(N_BOOTSTRAPS):
        idx = moving_block_indices(
            len(subset),
            BOOTSTRAP_BLOCK_LENGTH,
            rng,
        )
        sampled = subset.iloc[idx].copy()

        draws.append(
            {
                "iteration": iteration,
                "mean_pnl_usd": sampled["net_pnl_usd"].mean(),
                "mean_return_on_premium": sampled["net_return_on_premium"].mean(),
                "win_rate": (sampled["net_pnl_usd"] > 0).mean(),
                "profit_factor": profit_factor(sampled["net_pnl_usd"]),
            }
        )

    draw_frame = pd.DataFrame(draws)

    result = {
        "strategy": strategy,
        "trades": len(subset),
    }

    for metric in [
        "mean_pnl_usd",
        "mean_return_on_premium",
        "win_rate",
        "profit_factor",
    ]:
        low, high = ci(
            draw_frame[metric],
            CONFIDENCE_LEVEL,
        )
        result[f"{metric}_bootstrap_mean"] = draw_frame[metric].replace(
            [np.inf, -np.inf], np.nan
        ).mean()
        result[f"{metric}_ci_low"] = low
        result[f"{metric}_ci_high"] = high

    bootstrap_rows.append(result)

bootstrap_summary = pd.DataFrame(bootstrap_rows)
display(bootstrap_summary)

bootstrap_path = RQ5_TABLE_ROOT / "rq5_primary_bootstrap_uncertainty.csv"
bootstrap_summary.to_csv(bootstrap_path, index=False)
print("Saved:", bootstrap_path)

## 18. Conservative dissertation interpretation generator

The generated conclusion deliberately separates:

1. whether the strategy is profitable in the frictionless reference case;
2. whether it remains profitable under the primary medium-cost assumptions;
3. whether ML adds value over mean reversion;
4. whether farther-OTM contracts improve affordability at the cost of higher loss risk.

Review the wording before inclusion in the dissertation.

In [ ]:
def get_summary(strategy, offset, scenario):
    row = strategy_summary[
        strategy_summary["strategy"].eq(strategy)
        & strategy_summary["otm_offset_points"].eq(offset)
        & strategy_summary["execution_scenario"].eq(scenario)
    ]

    if len(row) != 1:
        return None

    return row.iloc[0]


ml_frictionless = get_summary(
    "RQ1_ML",
    PRIMARY_STRIKE_OFFSET,
    "frictionless",
)
ml_primary = get_summary(
    "RQ1_ML",
    PRIMARY_STRIKE_OFFSET,
    PRIMARY_COST_SCENARIO,
)
mr_primary = get_summary(
    "MeanReversion60m",
    PRIMARY_STRIKE_OFFSET,
    PRIMARY_COST_SCENARIO,
)

if ml_primary is None:
    raise ValueError("Primary ML ATM medium-cost result is unavailable.")

if ml_primary["total_pnl_usd"] > 0:
    cost_clause = (
        "The primary ML strategy remained profitable under the pre-specified "
        f"{PRIMARY_COST_SCENARIO} execution-cost assumptions"
    )
else:
    cost_clause = (
        "The primary ML strategy was not profitable after applying the "
        f"pre-specified {PRIMARY_COST_SCENARIO} execution-cost assumptions"
    )

if mr_primary is not None:
    if ml_primary["total_pnl_usd"] > mr_primary["total_pnl_usd"]:
        benchmark_clause = (
            "On the same opportunity dates and ATM contract specification, "
            "the ML direction generated greater cumulative P&L than the "
            "60-minute mean-reversion comparator"
        )
    else:
        benchmark_clause = (
            "On the same opportunity dates and ATM contract specification, "
            "the 60-minute mean-reversion comparator generated at least as much "
            "cumulative P&L as the ML direction"
        )
else:
    benchmark_clause = (
        "A complete paired comparison with the mean-reversion benchmark was "
        "not available because of missing option observations"
    )

afford_ml = affordability[
    affordability["strategy"].eq("RQ1_ML")
].copy()

if len(afford_ml):
    cheapest = afford_ml.sort_values(
        "median_entry_capital_usd"
    ).iloc[0]

    affordability_clause = (
        f"The lowest median entry-capital requirement among the tested strike "
        f"distances occurred at approximately "
        f"{int(cheapest['otm_offset_points'])} SPX points OTM "
        f"(${cheapest['median_entry_capital_usd']:.2f} median capital at risk). "
        f"Its near-total-loss rate was "
        f"{cheapest['near_total_loss_rate']:.1%}. "
        "This highlights the trade-off between affordability and the greater "
        "probability of substantial premium loss in farther-OTM 0DTE options."
    )
else:
    affordability_clause = (
        "The affordability comparison could not be completed because no "
        "usable strike-distance observations were available."
    )

frictionless_clause = ""

if ml_frictionless is not None:
    frictionless_clause = (
        f"In the frictionless reference case, the ATM ML strategy produced "
        f"total P&L of ${ml_frictionless['total_pnl_usd']:.2f} across "
        f"{int(ml_frictionless['trades'])} trades. "
    )

rq5_draft = f"""
### RQ5: 0DTE Options Trading Evaluation

The RQ5 backtest converted the previously developed RQ1–RQ3 opportunity
signals into same-day SPX option positions. Trades were restricted to sessions
meeting the pre-specified high-confidence RQ1, RQ2 predicted-magnitude, and
RQ3 large-movement conditions. The primary specification used the nearest
available ATM 0DTE contract, with RQ1 determining Call versus Put direction.

{frictionless_clause}{cost_clause}. Under this primary cost scenario, the ML
strategy completed {int(ml_primary['trades'])} trades, generated total P&L of
${ml_primary['total_pnl_usd']:.2f}, achieved a win rate of
{ml_primary['win_rate']:.1%}, and had mean return on premium of
{ml_primary['mean_return_on_premium']:.1%}. The corresponding profit factor
was {ml_primary['profit_factor']:.3f} and maximum dollar drawdown was
${abs(ml_primary['max_drawdown_usd']):.2f}.

{benchmark_clause}. This comparison is important because the simple
mean-reversion rule previously outperformed the machine-learning model on
unconditional directional balanced accuracy.

{affordability_clause}

Because historical NBBO quote data were unavailable under the Options Starter
dataset, exact historical bid-ask execution could not be reconstructed.
The results therefore use trade-derived minute aggregates together with
pre-specified adverse-execution and commission sensitivity scenarios.
Consequently, RQ5 should be concluded from the consistency of profitability
across these cost assumptions, the comparison against the mean-reversion
benchmark, and the stability of results across strike distances rather than
from the frictionless result alone.
"""

print(rq5_draft)

draft_path = RQ5_ROOT / "rq5_dissertation_draft.md"
draft_path.write_text(rq5_draft, encoding="utf-8")
print("Saved:", draft_path)

## 19. Save complete trade logs and manifest

In [ ]:
trade_log_path = RQ5_BACKTEST_ROOT / "rq5_complete_trade_log.parquet"
trade_csv_path = RQ5_BACKTEST_ROOT / "rq5_complete_trade_log.csv"

trades.to_parquet(trade_log_path, index=False)
trades.to_csv(trade_csv_path, index=False)

manifest = {
    "research_question": "RQ5",
    "fresh_holdout_used": False,
    "primary_rule": {
        "rq1_confidence_coverage": PRIMARY_RQ1_COVERAGE,
        "rq2_min_predicted_move_bps": PRIMARY_RQ2_MIN_BPS,
        "rq3_large_move_required": True,
    },
    "primary_direction_model": "RQ1_ML",
    "direction_benchmark": "60-minute mean reversion",
    "primary_strike_offset_points": PRIMARY_STRIKE_OFFSET,
    "strike_sensitivity_offsets_points": STRIKE_OFFSETS_POINTS,
    "entry_window_et": [ENTRY_WINDOW_START, ENTRY_WINDOW_END],
    "entry_reference": ENTRY_REFERENCE_FIELD,
    "exit_window_et": [EXIT_WINDOW_START, EXIT_WINDOW_END],
    "exit_reference": EXIT_REFERENCE_FIELD,
    "primary_execution_scenario": PRIMARY_COST_SCENARIO,
    "execution_scenarios": EXECUTION_SCENARIOS,
    "default_contract_multiplier": DEFAULT_CONTRACT_MULTIPLIER,
    "near_total_loss_return_threshold": NEAR_TOTAL_LOSS_RETURN,
    "affordability_thresholds_usd": AFFORDABILITY_THRESHOLDS_USD,
    "historical_nbbo_used": False,
    "notes": [
        "Trade-derived minute aggregates are used as reference prices.",
        "Synthetic adverse-execution scenarios are sensitivity assumptions, not reconstructed historical spreads.",
        "ATM remains the primary specification; OTM results are sensitivity analyses.",
        "RQ2 >=25 and >=30 bps results are sensitivity analyses only.",
    ],
}

manifest_path = RQ5_ROOT / "rq5_backtest_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, indent=2, default=str),
    encoding="utf-8",
)

print("Saved:")
for path in [
    trade_log_path,
    trade_csv_path,
    manifest_path,
]:
    print(" -", path)

## 20. Output inventory

The most important outputs for interpreting RQ5 are:

- `rq5_primary_atm_comparison.csv`
- `rq5_strategy_summary.csv`
- `rq5_retail_affordability.csv`
- `rq5_transaction_cost_sensitivity.csv`
- `rq5_rq2_threshold_sensitivity.csv`
- `rq5_paired_ml_vs_mean_reversion_summary.csv`
- `rq5_primary_bootstrap_uncertainty.csv`
- `rq5_dissertation_draft.md`
- `rq5_complete_trade_log.parquet`

### Interpretation order

1. **Primary ATM medium-cost result**
2. **ML versus mean-reversion benchmark**
3. **Transaction-cost sensitivity**
4. **Bootstrap uncertainty**
5. **OTM affordability sensitivity**
6. **RQ2 threshold sensitivity**
7. **Regime analysis**
8. **Fresh future holdout only after the model/trading rules remain frozen**